In [0]:
from pyspark.sql.functions import col, current_timestamp, to_json, struct, lit, trim, upper, when, split, element_at, size, expr, to_date
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

vendor_name = "opensignal"
layer_name = "silver_conform"

print(f"--- Starting Silver Conformance for: {vendor_name} ---")

# 1. Read directly from the Bronze table
df_bronze = spark.table("inlap.bronze.opensignal")

# 2. Coordinate parsing and canonical gold-join columns
df_cleaned = (
    df_bronze
    .withColumn(
        "coordinates_clean",
        when(col("coordinates").contains(";"), expr("replace(coordinates, ';', ',')")).otherwise(col("coordinates"))
    )
    .withColumn("coord_array", split(col("coordinates_clean"), ","))
    .withColumn(
        "latitude",
        when(size(col("coord_array")) == 2, trim(element_at(col("coord_array"), 1)).cast("double")).otherwise(lit(None).cast("double"))
    )
    .withColumn(
        "longitude",
        when(size(col("coord_array")) == 2, trim(element_at(col("coord_array"), 2)).cast("double")).otherwise(lit(None).cast("double"))
    )
    .withColumn("event_timestamp", col("measured_at").cast("timestamp"))
    .withColumn("event_date", to_date(col("measured_at")))
    .withColumn("site_ref", trim(col("site_ref")))
    .withColumn("network_type", upper(trim(col("network_type"))))
    .withColumn("signal_dbm", col("signal_dbm").cast("double"))
)

# 3. Define Quality Rules
valid_condition = (
    col("latitude").isNotNull()
    & col("longitude").isNotNull()
    & ((col("latitude") != 0.0) | (col("longitude") != 0.0))
    & col("site_ref").isNotNull()
    & (col("site_ref") != "SITE-UNKNOWN")
    & col("signal_dbm").isNotNull()
)

# 4. Split into Valid vs Quarantined
df_passed_dq = df_cleaned.filter(valid_condition)
df_dq_quarantine = df_cleaned.filter(~valid_condition) \
    .drop("coord_array", "coordinates_clean") \
    .withColumn(
        "failure_reason",
        lit("malformed coordinate string, missing signal, or invalid/placeholder site reference")
    )

# 5. Handle duplicates among valid records (keep latest reading per site_ref)
window_spec = Window.partitionBy("site_ref").orderBy(col("event_timestamp").desc())
df_with_rn = df_passed_dq.withColumn("row_num", row_number().over(window_spec))

df_valid = df_with_rn.filter(col("row_num") == 1).drop("row_num", "coord_array", "coordinates_clean")
df_dup_quarantine = df_with_rn.filter(col("row_num") > 1) \
    .drop("row_num", "coord_array", "coordinates_clean") \
    .withColumn("failure_reason", lit("duplicate site_ref reading"))

# 6. Combine all quarantine records and structure final output
df_all_quarantine = df_dq_quarantine.unionByName(df_dup_quarantine)
df_quarantine_final = df_all_quarantine \
    .drop("row_num") \
    .withColumn("source_vendor", lit(vendor_name)) \
    .withColumn("quarantine_timestamp", current_timestamp()) \
    .withColumn("raw_record", to_json(struct([col(c) for c in df_bronze.columns]))) \
    .select("raw_record", "failure_reason", "source_vendor", "quarantine_timestamp")

# 7. Refresh Silver Conformed and append quarantine rows
df_valid.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("inlap.silver.opensignal_conformed")

df_quarantine_final.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("inlap.silver.quarantine_records")

# 8. Log Audit Metrics
rows_read = df_bronze.count()
rows_passed = df_valid.count()
rows_quarantined = df_quarantine_final.count()

spark.sql(f"""
    INSERT INTO inlap.control.audit_log 
    VALUES (
        '{vendor_name}', 
        '{layer_name}', 
        current_timestamp(), 
        'SUCCESS', 
        {rows_read}, 
        {rows_passed}, 
        {rows_quarantined}
    )
""")

print(f"Conformance complete for {vendor_name}. Read: {rows_read} | Passed: {rows_passed} | Quarantined: {rows_quarantined}")